In [1]:
from Bio import SeqIO
from Bio.Seq import Seq

def complement(seq):
    return seq.upper()


ltg = 'taagta'
tc = complement('ga')
rtg = 'cgccta'

ldg = 'tattgtcata'
dc = complement('ga')
rdg = 'acataatagg'

extra_seq = 'RRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRR'
target_context = "RRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRR"

In [2]:
product = complement(ltg) + dc + rdg + complement(extra_seq) + complement(ldg) + tc + rtg + target_context
product = complement(ltg).upper() + dc + rdg + complement(extra_seq).upper() + complement(ldg).upper() + tc + rtg + target_context
print(product)
# attcatctacataatagg

TAAGTAGAacataataggRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRTATTGTCATAGAcgcctaRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRR


In [3]:
# Example from Fig. 5 of structure paper
lt = 'ATCAGGC'
tc = 'CT'
rt = 'AC'
# rt = 'TG'

ld = 'ACAGTAT'
dc = 'CT'
rd = 'TGTAT'
# rd = 'ACATA'

extra_seq = 'ARRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRY'
target_context = "RRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRR"
product = (complement(lt) 
           + " " + dc +  " " + rd + " " 
           + complement(extra_seq) +  " " 
           + complement(ld) + " " + tc + " " + rt + " " + target_context)
print(product)


ATCAGGC CT TGTAT ARRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRY ACAGTAT CT AC RRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRR


In [4]:
print(f"Target: {lt + tc + rt}")
print(f"Donor: {ld + dc + rd}")

Target: ATCAGGCCTAC
Donor: ACAGTATCTTGTAT


In [5]:
ts_contig = SeqIO.parse("/groups/cress/projects/is110/sequences/SLMQRT/monomeric_BAC_for_plate_scrape_13kb.gbk", "genbank")
for record in ts_contig:
    seq = record.seq
complement(seq)

Seq('ATGAAACTCATGGAAACACTTAACCAGTGCATAAACGCTGGTCATGAAATGACG...AGA')

In [6]:
import argparse


# 5' ----- LT - target core - RT ----- 3' 
# 5' ----- LD - donor core - RD ----- 3' 


parser = argparse.ArgumentParser(
    description = "Script to predict IS110 recombination product given target "
    "and donor contigs, left and right ends of target and donor sites, and " \
    "cores of each. "
)
parser.add_argument("-tg", "--target_contig", help="Path to the genome/contig with target site")
parser.add_argument("-dg", "--donor_contig", help="Path to the genome/contig with donor site", required=False)


parser.add_argument("-lt", "--left_target", help="",)
parser.add_argument("-rt", "--right_target", help="",)
parser.add_argument("-ld", "--left_donor", help="",)
parser.add_argument("-rd", "--right_donor", help="",)
parser.add_argument("-tc", "--target_core", default="CT",
                    help="",)
parser.add_argument("-dc", "--donor_core", required=False,
                    help="",)





args = parser.parse_args(
    [
        "--target_contig", "/groups/cress/projects/is110/sequences/SLMQRT/Manually_Adjusted_Seq_of_HalfKan_Reporter_Strain_SLMQRT_4_dnaSES0085_no_BAC.gbk",
        "--donor_contig", "/groups/cress/projects/is110/sequences/SLMQRT/monomeric_BAC_for_plate_scrape_13kb.gbk",
        "-lt", "attcat",
        "-tc", "ct",
        "-rt", "gcggat",
        "-ld", "CCTATTATGTT",
        "-dc", "CT",
        "-rd", "ATGACAATA", # NOTE: not sure if this donor seq is correct
        # "-tc", "CT",
        
        
        # ""
        ])



same_contig = False

if not args.donor_core:
    args.donor_core = args.target_core
    print(f"Assuming donor core = target core = {args.target_core}")
else:
    print(f"donor core = {args.donor_core}; target core = {args.target_core}")


if not args.donor_contig:
    print("No donor contig provided. Assuming donor and target on same contig...")
    # Need logic to look for donor on target contig
    same_contig = True
    ...
    # If target is on one strand and donor is on the other, will need to complement all elements of donor.




lt = args.left_target.upper()
rt = args.right_target.upper()
ld = args.left_donor.upper()
rd = args.right_donor.upper()
tc = args.target_core.upper()
dc = args.donor_core.upper()

target_site = lt + tc + rt
donor_site = ld + dc + rd





# Read sequences of target and donor files
print("Reading target contig...", end=" ")
target_contig_records = list(SeqIO.parse(args.target_contig, "genbank"))
assert len(target_contig_records) == 1, "Found more than one record in target contig file!"

for record in target_contig_records:
    target_contig = str(Seq(record.seq))
    print("done")

print("Reading donor contig...", end=" ")
donor_contig_records = list(SeqIO.parse(args.donor_contig, "genbank"))
assert len(donor_contig_records) == 1, "Found more than one record in donor contig file!"

for record in donor_contig_records:
    donor_contig = str(Seq(record.seq))
    print("done")


donor core = CT; target core = ct
Reading target contig... done
Reading donor contig... done


In [7]:
import re

# [match for match in re.finditer("A", "AAA")]
def complement(seq):
    return str(Seq(seq).complement())


def find_matches(site, contig, complement_search=True, return_contig=True):
    """
    Find matches for site in contig. If none found, look for it in the complement
    of the contig instead if complement_search=True.
    """
    matches_in_contig = [match for match in re.finditer(site, contig)]
    if (len(matches_in_contig) > 1):
        raise ValueError("Found multiple sites in contig!")
    elif (len(matches_in_contig) == 0):
        if complement_search:
            print("\tsearching for site in complement...")
            contig = complement(contig)
            matches_in_contig = [match for match in re.finditer(site, contig)]
            if (len(matches_in_contig) > 1):
                raise ValueError("Found multiple sites in contig!")
            elif (len(matches_in_contig) == 0):
                raise ValueError("Found 0 sites in contig!")
        else:
            raise ValueError("Found 0 sites in contig!")

    site_start, site_end = matches_in_contig[0].start(), matches_in_contig[0].end()
    # contig[site_start:site_end] == site
    # NOTE: safe to index like [site_end:] or [:site_start] to extract contig 
    if return_contig:
        return contig, site_start, site_end
    else:
        return site_start, site_end


print("Finding target site in target contig")
target_contig, target_start, target_stop = find_matches(target_site, target_contig)
print("Finding donor site in donor contig")
donor_contig, donor_start, donor_stop = find_matches(donor_site, donor_contig)

    # raise ValueError("Found 0 target sites in target contig!")

Finding target site in target contig
	searching for site in complement...
Finding donor site in donor contig
	searching for site in complement...


In [8]:
def find_partial_site(part, site, contig_pos):
    """_summary_

    Args:
        part (str): Part of site. e.g. LT, RT, LD, RD
        site (str): Full sequence of target or donor site
        contig_pos (str): Start index of site in contig
    """    
    part_start, part_end = find_matches(part, site, complement_search=False, return_contig=False)
    part_start, part_end = contig_pos + part_start, contig_pos + part_end
    return part_start, part_end

# Extract indices of LT, RT, LD, RD
lt_start, lt_end = find_partial_site(lt, target_site, target_start)
rt_start, rt_end = find_partial_site(rt, target_site, target_start)
ld_start, ld_end = find_partial_site(ld, donor_site, donor_start)
rd_start, rd_end = find_partial_site(rd, donor_site, donor_start)


In [9]:
# Get sequences of donor + cargo besides target / donor site
target_external_seq = target_contig[target_stop:] + target_contig[:target_start]
donor_cargo_seq = donor_contig[donor_stop:] + donor_contig[:donor_start]


In [10]:
# if not same_contig:
result = (complement(lt) + tc + rd 
          + complement(donor_cargo_seq) 
          + complement(ld) + dc + rt + target_external_seq)

print(result[:10])
print(len(result))

TAAGTACTAT
4647701


In [44]:
import pandas as pd
import numpy as np

result_parts = [complement(lt), tc, rd, complement(donor_cargo_seq), complement(ld), dc, rt, target_external_seq]
result = "".join(result_parts)

result_part_names = ["LT", "Core", "RD", "Insertion Sequence", "LD", "Core", "RT", "Target Contig"]
target_color = "#00aeff"
target_contig_color = "#a5e3ff"
donor_color = "#ff8400"
donor_contig_color = "ffc484"
core_color = "#24951a"

part_colors = [target_color, core_color, donor_color, donor_contig_color, donor_color, core_color, target_color, target_contig_color]


part_lens = [len(i) for i in result_parts]
part_ends = np.cumsum(part_lens)
part_starts = part_ends - part_lens #+ 1 # 1-based indexing?

parts_df = pd.DataFrame({
    "seq": result_parts, 
    "name": result_part_names, 
    "color": part_colors, 
    "len": part_lens,
    "start": part_starts,
    "end": part_ends,
    
    })

display(parts_df)

features = []

for _, row in parts_df.iterrows():
    location = FeatureLocation(row["start"], row["end"]) # Add strand?
    feat = SeqFeature(location=location, type="misc_feature", qualifiers = {
        "label": row["name"],
        "ApEinfo_fwdcolor": row["color"],
        "ApEinfo_revcolor": row["color"],
    })
    features.append(feat)


# Format the result as a genbank file
result_id = "recombined product" # I'll fill this in later.
result_name = "recombined_product" # I'll fill this in later.
result_description = f"Predicted recombination product using target site = {target_site} and donor site = {donor_site}"

recombinant_record = SeqRecord(Seq(result), id=result_id, name=result_name, description=result_description)
# Mark the record as circular
print("NOTE: assuming circular topology")
recombinant_record.annotations["topology"] = "circular"
recombinant_record.annotations["molecule_type"] = "DNA"
# Apply features to genbank record and save file
recombinant_record.features = features
with open("../misc/recombinant_test.gb", "w") as output_handle:
    SeqIO.write(recombinant_record, output_handle, "genbank")


,seq,name,color,len,start,end
0,TAAGTA,LT,#00aeff,6,0,6
1,CT,Core,#24951a,2,6,8
2,ATGACAATA,RD,#ff8400,9,8,17
3,CGCCGTCGGGCATGCGCGCCTTGAGCCTGGCGAACAGTTCGGCTGG...,Insertion Sequence,ffc484,13733,17,13750
4,GGATAATACAA,LD,#ff8400,11,13750,13761
5,CT,Core,#24951a,2,13761,13763
6,GCGGAT,RT,#00aeff,6,13763,13769
7,AGTCGCTAGCCGAGCAACGGGACGCGGCGAGGTTTCGGGCGCTGCG...,Target Contig,#a5e3ff,4633932,13769,4647701


NOTE: assuming circular topology


In [69]:
comp = lambda s: print(Seq(s).complement())
comp("cctattatgttctgcggat")
comp("attcatctatgacaata")
comp("ggtacccagtgctgctctaggagattcatctatgacaatagcggcagcccgta")

ggataatacaagacgccta
taagtagatactgttat
ccatgggtcacgacgagatcctctaagtagatactgttatcgccgtcgggcat


In [61]:
s = Seq("tcagaagaactcgtcaagaaggcgatagaaggcgatgcgctgcgaatcgggagcggcgataccgtaaagcacgaggaagcggtcagcccattcgccgccaagctcttcagcaatatcacgggtagccaacgctatgtcctgatagcggtccgccacacccagccggccacagtcgatgaatccagaaaagcggccattttccaccatgatattcggcaagcaggcatcgccatgggtcacgacgagatcctc")
print(s.reverse_complement())

gaggatctcgtcgtgacccatggcgatgcctgcttgccgaatatcatggtggaaaatggccgcttttctggattcatcgactgtggccggctgggtgtggcggaccgctatcaggacatagcgttggctacccgtgatattgctgaagagcttggcggcgaatgggctgaccgcttcctcgtgctttacggtatcgccgctcccgattcgcagcgcatcgccttctatcgccttcttgacgagttcttctga


In [ ]:
new_location = FeatureLocation(new_start, new_end, strand=feat.location.strand)
new_feat = SeqFeature(location=new_location, type=feat.type, qualifiers=feat.qualifiers)
transferred_features.append(new_feat)

In [72]:
def reverse_complement(seq):
    return str(Seq(seq).reverse_complement())

def find_matches(site, contig, complement_search=True, return_contig=True):
    """
    Find matches for site in contig. If none found, look for it in the complement
    of the contig instead if complement_search=True.
    """
    matches_in_contig = [match for match in re.finditer(site, contig)]
    if (len(matches_in_contig) > 1):
        raise ValueError("Found multiple sites in contig!")
    elif (len(matches_in_contig) == 0):
        if complement_search:
            print("\tsearching for site in reverse complement...", end=" ")
            contig = reverse_complement(contig)
            matches_in_contig = [match for match in re.finditer(site, contig)]
            if (len(matches_in_contig) > 1):
                raise ValueError("Found multiple sites in contig!")
            elif (len(matches_in_contig) == 0):
                raise ValueError("Found 0 sites in contig!")
            else:
                print("found!")
        else:
            raise ValueError("Found 0 sites in contig!")

    site_start, site_end = matches_in_contig[0].start(), matches_in_contig[0].end()
    # contig[site_start:site_end] == site
    # NOTE: safe to index like [site_end:] or [:site_start] to extract contig 
    if return_contig:
        return contig, site_start, site_end
    else:
        return site_start, site_end

ctg = "AAAAATTAAAAAGGAAAAA"
ts = "TT"
ds = "CC"

find_matches(ts, ctg), find_matches(ds, ctg)

	searching for site in reverse complement... found!


(('AAAAATTAAAAAGGAAAAA', 5, 7), ('TTTTTCCTTTTTAATTTTT', 5, 7))

In [73]:
complement("ATTCATCTGCGGAT")

'TAAGTAGACGCCTA'

# ChatGPT code for transferring features between OG gb file and new one

In [31]:
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature, FeatureLocation

result_id = "recombined product" # I'll fill this in later.
result_name = "recombined_product" # I'll fill this in later.
result_description = f"Predicted recombination product using target site = {target_site} and donor site = {donor_site}"

recombinant_record = SeqRecord(Seq(result), id=result_id, name=result_name, description=result_description)
# Mark the record as circular
recombinant_record.annotations["topology"] = "circular"
recombinant_record.annotations["molecule_type"] = "DNA"

In [33]:
original = target_contig_records[0]
# FUNC DEF'N!



transferred_features = []
for feat in original.features[:10]:
    # Only consider features with defined location and sequence
    try:
        feat_seq = feat.extract(original.seq)
    except:
        continue

    # Convert feature sequence to string for searching
    feat_seq_str = str(feat_seq)

    # Search for exact match in the recombinant sequence
    match = re.search(re.escape(feat_seq_str), result)
    if match:
        new_start = match.start()
        new_end = match.end()

        new_location = FeatureLocation(new_start, new_end, strand=feat.location.strand)
        new_feat = SeqFeature(location=new_location, type=feat.type, qualifiers=feat.qualifiers)
        transferred_features.append(new_feat)



In [36]:
# new_feat.qualifiers
feat.qualifiers

{'gene': ['recF'],
 'locus_tag': ['OCPLJD_00004'],
 'label': ['recF'],
 'note': ['color: #a6acb3; direction: RIGHT']}

In [ ]:
recombinant_record.annotations["topology"] = "circular"
recombinant_record.annotations["molecule_type"] = "DNA"
recombinant_record.features = transferred_features
with open("../misc/recombinant_test.gb", "w") as output_handle:
    SeqIO.write(recombinant_record, output_handle, "genbank")

In [ ]:
# # Load original GenBank
# original = SeqIO.read("original.gb", "genbank")

# # Simulated recombinant sequence as string
# recombinant_seq = "ATGC..."  # replace with your actual string

# # Convert to Biopython SeqRecord
# recombinant_record = SeqRecord(Seq(recombinant_seq), id="recombinant", description="Generated from sequence string")

# # Prepare to hold transferred features
# transferred_features = []

# # Search for each original feature sequence in the recombinant
# for feat in original.features:
#     # Only consider features with defined location and sequence
#     try:
#         feat_seq = feat.extract(original.seq)
#     except:
#         continue

#     # Convert feature sequence to string for searching
#     feat_seq_str = str(feat_seq)

#     # Search for exact match in the recombinant sequence
#     match = re.search(re.escape(feat_seq_str), recombinant_seq)
#     if match:
#         new_start = match.start()
#         new_end = match.end()

#         new_location = FeatureLocation(new_start, new_end, strand=feat.location.strand)
#         new_feat = SeqFeature(location=new_location, type=feat.type, qualifiers=feat.qualifiers)
#         transferred_features.append(new_feat)

# # Assign to the new record
# recombinant_record.features = transferred_features

# # Save to GenBank
# SeqIO.write(recombinant_record, "recombinant.gb", "genbank")